In [ ]:
import pandas as pd
import random
import re


In [ ]:
from pathlib import Path
BASE_DIR = Path().resolve().parent
file_path = BASE_DIR / "data" / "dataset.csv"


---
# SHARED POOLS
# Names, companies, seniority and process vocabulary — sampled
# independently of every label, so none can become a spurious predictor.
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHARED POOLS
# ══════════════════════════════════════════════════════════════════════════════
# Everything here is sampled INDEPENDENTLY of category and job_field.
#
# Names were a measured leak in the previous dataset: the trained model's top
# features for 'general' included "conversation yamamoto" and "link sato" —
# randomly assigned candidate names that correlated with a class by chance.
#
# Companies were the same leak with a real cause: the old generator gave each
# field its own company list, so "hugging face" predicted data_science and
# "atlassian" predicted software_engineering. Real companies hire across every
# field, and a model keyed on company names cannot generalise to a company it
# has not seen. One pool, no field association.
#
# Seniority and process vocabulary live here for a different reason: they are
# real recruiting vocabulary that was entirely missing (internship, new grad,
# hiring manager, take-home all appeared ZERO times), but they are not FIELD
# signal. "intern" already appeared in 38 rows spread evenly across four fields
# and carried no field information. Shared pools add the vocabulary without
# manufacturing a spurious predictor.

first_names = [
    "Sarah", "James", "Emily", "Michael", "Sophie", "David", "Rachel", "Daniel",
    "Ayşe", "Mehmet", "Elif", "Emre", "Fatma", "Ilker", "Zeynep", "Burak",
    "Priya", "Arjun", "Ananya", "Rahul", "Neha", "Vikram", "Divya", "Karthik",
    "Wei", "Li", "Xiao", "Mei", "Jian", "Ling", "Chen", "Yan",
    "Min-jun", "Ji-woo", "Seo-yeon", "Hyun-woo", "Soo-jin", "Jae-hyun",
    "Haruto", "Yuki", "Sakura", "Ren", "Aoi", "Takumi",
    "Carlos", "Maria", "Juan", "Isabella", "Luis", "Camila", "Diego", "Valentina",
    "Aisha", "Omar", "Layla", "Yusuf", "Zara", "Hassan", "Nadia", "Karim",
    "Kwame", "Amara", "Chidi", "Ngozi", "Tunde", "Folake",
]

recruiter_names = [
    "John Baker", "Lisa Moreau", "Rachel Kim", "Tom Fletcher", "Nina Alvarez",
    "Kevin Osei", "Julia Brandt", "Marcus Webb", "Sofia Rinaldi", "Daniel Novak",
    "Hannah Cole", "Peter Lindqvist", "Grace Adeyemi", "Oliver Hart",
    "Maya Krishnan", "Ethan Blackwood", "Clara Dubois", "Sam Whitfield",
    "Ines Ferreira", "Jonas Weber", "Amira Haddad", "Ryan Doherty",
    "Petra Kovacs", "Noah Bergman", "Yuna Park", "Alex Renner",
]

# One pool. A company never implies a field.
#
# Deliberately FIELD-NEUTRAL proper nouns. An earlier draft used names like
# "Kestrel Data", "Quarry Software", "Sable Analytics" and "Anvil Cloud", which
# scattered the words data/software/analytics/cloud at random across every field
# — injecting the exact tokens the job_field head relies on into rows that have
# nothing to do with them. Keep new entries free of technology words.
companies = [
    "Northwind", "Redwood", "Harborstone", "BluePeak", "Lumenary",
    "Cobalt", "Vantage", "Ironbridge", "Silverline", "Nimbus",
    "Arcadia", "Meridian", "Crestwave", "Foundry Nine",
    "Brightpath", "Orchard", "Stonebridge", "Halcyon",
    "Peregrine", "Tessellate", "Kestrel", "Anvil", "Waypoint",
    "Bramble", "Clearwater", "Junction", "Solstice", "Ridgeline",
    "Keystone", "Marlow & Co", "Quarry", "Sable",
    "Thornbury", "Upland", "Verdant", "Wexler",
    "Ashgrove", "Beacon", "Calderon", "Drayton",
    "Evergreen", "Fernbrook", "Granite Peak", "Hollowell",
    "Inlet", "Juniper Grove", "Larkspur", "Millbrook",
    "Oakhaven", "Pinehurst",
]

sender_titles = [
    "Technical Recruiter", "Talent Acquisition Partner", "Recruiting Coordinator",
    "Hiring Manager", "Engineering Manager", "Head of Talent", "People Partner",
    "Talent Partner", "University Recruiter", "Senior Recruiter",
    "Talent Sourcer", "Recruitment Lead",
]

# Who the candidate will meet. Generic on purpose — no field words.
interviewer_refs = [
    "the hiring manager", "our engineering manager", "two members of the team",
    "the team lead", "one of our senior team members", "the department head",
    "a panel of three", "our technical lead", "the team you'd be joining",
    "myself and the hiring manager",
]

# Seniority / stage words. Real recruiting vocabulary, NOT field signal.
seniority_phrases = [
    "internship", "summer internship", "12-week internship", "new grad role",
    "new graduate position", "new grad opening", "new grad programme",
    "entry level position", "junior position",
    "mid-level position", "senior position", "graduate programme",
    "co-op placement", "placement year role", "returning internship",
    "full-time position", "contract role", "permanent position",
]

# Process vocabulary. Shared for the same reason.
process_terms = [
    "technical screen", "phone screen", "initial screen", "take-home exercise",
    "onsite", "virtual onsite", "panel interview", "final round", "first round",
    "second round", "HR round", "culture fit conversation", "hiring committee",
    "background check", "reference check", "applicant tracking system",
    "referral", "offer stage", "debrief",
]

platforms = [
    "Zoom", "Google Meet", "Microsoft Teams", "Google Hangouts", "Zoom (link to follow)",
    "our video platform", "Teams", "a Zoom call", "Google Meet (invite attached)",
]

durations = [
    "30 minutes", "45 minutes", "an hour", "60 minutes", "90 minutes",
    "about 45 minutes", "roughly an hour", "two hours", "half a day",
]

greetings = ["Hi", "Hello", "Dear", "Hey", "Good morning", "Good afternoon"]

signoffs = [
    "Best", "Best regards", "Kind regards", "Regards", "Many thanks",
    "Thanks", "All the best", "Warm regards", "Sincerely", "Cheers",
]

# Category-NEUTRAL courtesy lines. These bring the median length up to the 75-90
# words real email actually runs at. They are shared across all seven categories
# precisely because they carry no label signal — a line that works equally in a
# rejection and an offer cannot help the model predict either. The leak check
# (no shared-pool token in any class's top-15 coefficients) verifies this rather
# than assuming it.
closing_lines = [
    "Please let me know if you have any questions.",
    "Do get in touch if anything is unclear.",
    "I'm happy to answer anything over email.",
    "Thanks again for your interest in {company}.",
    "Thanks for your time.",
    "Feel free to reply directly to this message.",
    "Any questions, just ask.",
    "I appreciate you taking the time to read this.",
]

# These must read correctly in a REJECTION and an OFFER as well as an invitation.
# An earlier draft included "I'll keep you updated at each stage" and "please add
# us to your contacts" — forward-looking process language that contradicts a
# rejection outright. Dropping them into terminal categories was measurably
# harmful: rejection recall fell to 0.47, with 21 rejections read as
# application_received. Anything implying the process continues does not belong
# in a globally shared pool.
courtesy_lines = [
    "You can find more about the team and how we work on our careers page.",
    "Contact details are below if you'd rather speak by phone.",
    "We know job hunting takes time, and we try to keep our process short.",
    "This address is monitored during business hours on weekdays.",
    "Everything we send is also visible in your candidate portal.",
    "My colleagues in the recruitment team can also help if I'm away.",
    "Our offices are closed on public holidays, which can affect response times.",
    "You're welcome to reply to this address with anything you need.",
]

# Category-neutral frames that carry FIELD vocabulary. Only a minority of
# template bodies reference {tech}, which left individual technologies with very
# few occurrences each — "python" landed in 10 rows, barely better than the zero
# it had before. These read naturally in any of the seven categories, so they add
# technology frequency without becoming a category signal.
tech_context_lines = [
    "The team works primarily with {tech} and {tech2}.",
    "Our stack is mostly {tech}.",
    "The role involves {tech} day to day.",
    "Familiarity with {tech} is part of the role.",
    "The codebase is largely {tech}, with some {tech2}.",
    "Most of the team's work is in {tech}.",
    "You'd be working with {tech} alongside {tech2}.",
    "The platform is built on {tech}.",
]

# ── urgency evidence ──
# The urgency label is DERIVED from one of these sentences, not drawn beside it.
# Previously urgency was random.choices(urgency_weights[category]) — assigned by
# category and recoverable from the text by nobody. A majority-per-category
# baseline scored 0.781 against a trained 0.758: the head had learned nothing and
# was fitting candidate names, which is why it was the only head still leaking.
#
# Phrasing is category-agnostic so a marker carries URGENCY and not category.
urgency_markers = {
    "high": [
        "Could you let me know by {deadline}? We're holding things until then.",
        "We'd need to hear back within 48 hours.",
        "This one is time-sensitive, so a quick reply would really help.",
        "Please confirm by {deadline} if you can.",
        "Apologies for the short notice — a reply by {deadline} would be ideal.",
        "At your earliest convenience, please.",
        "We need an answer by {deadline}.",
        "Could you come back to me today or tomorrow?",
    ],
    "medium": [
        "Let me know within the next week or so.",
        "No particular rush, but a reply in the next couple of weeks would help.",
        "Whenever you get a chance over the next fortnight.",
        "Please reply when it's convenient.",
        "Have a think and come back to me in the next week or two.",
        "There's no hard deadline, though sooner makes scheduling easier.",
        "Get back to me once you've had time to consider it.",
        "Sometime in the next two weeks is fine.",
    ],
    "low": [
        "No rush at all — reply whenever suits you.",
        "There's no deadline on this.",
        "No action is needed from you right now.",
        "Nothing needed from you at this stage.",
        "No pressure either way.",
        "This is just to keep you in the loop.",
        "Reply only if you'd like to; there's no obligation.",
        "You don't need to do anything with this.",
    ],
}

# Near-term deadlines for {deadline}. Kept separate from `dates`, which spans
# months — a HIGH marker must name something inside about a week.
near_deadlines = [
    "tomorrow", "Friday", "the end of the week", "Monday",
    "close of play tomorrow", "end of day Thursday", "this Wednesday", "Tuesday",
]

dates = [
    "Monday, March 3", "Tuesday, March 11", "Wednesday, April 2", "Thursday, April 17",
    "Friday, May 9", "Monday, May 19", "Tuesday, June 3", "Wednesday, June 18",
    "Thursday, July 10", "Friday, July 25", "Monday, August 4", "Tuesday, August 19",
    "Wednesday, September 3", "Thursday, September 11", "Friday, October 2",
    "Monday, October 20", "Tuesday, November 4", "Wednesday, November 19",
    "next Tuesday", "next Thursday", "the week of March 10", "the week of June 16",
    "early next week", "the following Monday", "March 14, 2026", "June 5, 2026",
    "September 22, 2026", "October 8, 2026",
]

times = [
    "9:00 AM", "9:30 AM", "10:00 AM", "10:30 AM", "11:00 AM", "11:30 AM",
    "1:00 PM", "1:30 PM", "2:00 PM", "2:30 PM", "3:00 PM", "3:30 PM",
    "4:00 PM", "4:30 PM", "5:00 PM", "10:00 AM CET", "2:00 PM EST",
    "11:00 AM PST", "9:00 AM GMT", "3:00 PM BST",
]

locations = [
    "our London office", "the Berlin office", "our Amsterdam HQ", "the Dublin office",
    "our New York office", "the Austin office", "our Toronto office",
    "the Manchester office", "our Istanbul office", "the Stockholm office",
]


---
# FIELD VOCABULARY — 11 tech classes
# 'general' is deliberately removed. Below the confidence threshold the UI
# shows "Unclassified"; that is a threshold decision, not a class.
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIELD VOCABULARY — 11 tech classes, no 'general'
# ══════════════════════════════════════════════════════════════════════════════
# 'general' is deliberately GONE. It was generated by BLANKING role and company,
# so it occupied the template skeleton rather than a semantic region and absorbed
# anything out-of-vocabulary — 0.37 precision / 0.78 recall, a magnet class that
# swallowed four consecutive software-engineering emails. Below the confidence
# threshold the UI shows "Unclassified"; that is a threshold decision, not a
# class, and it must not compete for probability mass here.
#
# The SAME template is used for all 11 fields. Field signal enters ONLY through
# the vocabulary substituted below, which is what forces the field head to learn
# terminology rather than phrasing — and why grouping by template leaks no field
# information.
#
# SEPARABILITY RULE for the three that overlap:
#   software_engineering — generic/backend role. MAY name no technology at all.
#   frontend             — every role or tech token is frontend-specific.
#   fullstack            — every role carries an EXPLICIT marker (full stack /
#                          fullstack / MERN), never merely backend + frontend
#                          tokens, which would make it an undecidable superset.

job_fields = {
    "software_engineering": {
        "roles": [
            "Software Engineer", "Software Development Engineer", "SDE", "SDE II",
            "Backend Engineer", "Backend Developer", "Server-Side Engineer",
            "API Engineer", "Software Engineer Intern", "SWE Intern",
            "Junior Software Engineer", "Senior Software Engineer",
            "Staff Software Engineer", "Software Engineer II", "Backend Software Engineer",
        ],
        "tech": [
            "Java", "Python", "Go", "Golang", "C#", "C++", "Ruby", "Rust",
            "Kotlin", "Scala", "Node.js", "Spring Boot", "Django", "Flask",
            "FastAPI", "Express", ".NET", "Rails", "PostgreSQL", "MySQL",
            "Redis", "MongoDB", "Kafka", "RabbitMQ", "gRPC", "GraphQL",
        ],
        "topics": [
            "system design", "data structures and algorithms", "API design",
            "database schema design", "concurrency", "scalability",
            "microservices architecture", "caching strategies",
            "LeetCode-style problems", "distributed systems", "backend architecture",
            "REST API design", "query optimisation", "service reliability",
        ],
        "formats": [
            "technical phone screen", "live coding round", "system design round",
            "take-home assignment", "onsite loop", "pair-programming exercise",
            "algorithms interview", "architecture discussion",
        ],
    },
    "frontend": {
        "roles": [
            "Frontend Engineer", "Front-End Developer", "Frontend Developer",
            "UI Engineer", "React Developer", "Frontend Intern",
            "Junior Frontend Developer", "Senior Frontend Engineer",
            "UI Developer", "Front End Engineer", "Web Frontend Engineer",
        ],
        "tech": [
            "JavaScript", "TypeScript", "HTML", "CSS", "SCSS", "SASS", "React",
            "Next.js", "Vue", "Vue.js", "Nuxt", "Angular", "Svelte", "Redux",
            "Tailwind CSS", "styled-components", "Webpack", "Vite", "Storybook",
            "Jest", "React Testing Library", "Chrome DevTools",
        ],
        "topics": [
            "responsive design", "accessibility", "a11y", "WCAG compliance",
            "component library work", "design system implementation",
            "state management", "browser rendering", "Core Web Vitals",
            "cross-browser support", "DOM manipulation", "bundle size optimisation",
            "component architecture", "CSS layout",
        ],
        "formats": [
            "component build exercise", "live coding in CodeSandbox", "UI take-home",
            "browser-based coding round", "frontend system design",
            "styling and layout exercise",
        ],
    },
    "fullstack": {
        "roles": [
            "Full Stack Engineer", "Full-Stack Developer", "Fullstack Engineer",
            "Full Stack Web Developer", "MERN Developer", "Full Stack Intern",
            "Junior Full Stack Developer", "Senior Full Stack Engineer",
            "Full Stack Software Engineer", "Fullstack Developer",
        ],
        "tech": [
            "React and Node", "Next.js", "Django and React", "Rails", "Laravel",
            "the MERN stack", "the MEAN stack", "Express with Prisma", "tRPC",
            "Node.js with PostgreSQL", "Vue with Laravel", "React with FastAPI",
            "TypeScript across the stack", "Next.js with Prisma",
        ],
        "topics": [
            "end-to-end ownership", "both the front and back end",
            "shipping features end to end", "the full product lifecycle",
            "API plus interface work", "building a CRUD application",
            "database through to UI", "full-stack feature delivery",
            "owning a feature from schema to screen",
        ],
        "formats": [
            "full-stack take-home", "build a small app exercise",
            "end-to-end feature round", "full-stack pairing session",
            "API and UI exercise",
        ],
    },
    "mobile": {
        "roles": [
            "iOS Engineer", "Android Engineer", "Mobile Engineer", "iOS Developer",
            "Android Developer", "React Native Developer", "Flutter Developer",
            "Mobile Intern", "Senior iOS Engineer", "Junior Android Developer",
            "Mobile Software Engineer", "iOS Engineer Intern",
        ],
        "tech": [
            "Swift", "SwiftUI", "Objective-C", "Kotlin", "Jetpack Compose",
            "React Native", "Flutter", "Dart", "UIKit", "Core Data", "Room",
            "Xcode", "Android Studio", "TestFlight", "Firebase",
        ],
        "topics": [
            "app lifecycle", "offline support", "push notifications",
            "App Store review", "mobile performance", "memory management",
            "device fragmentation", "background tasks", "deep linking",
            "mobile app architecture", "Play Store releases", "UI responsiveness on device",
        ],
        "formats": [
            "mobile coding exercise", "app architecture discussion",
            "device debugging round", "mobile take-home", "live coding on device",
        ],
    },
    "data_related": {
        "roles": [
            "Data Analyst", "Data Scientist", "Data Engineer", "Analytics Engineer",
            "Business Intelligence Analyst", "BI Analyst", "Junior Data Analyst",
            "Data Science Intern", "Data Engineering Intern", "Senior Data Analyst",
            "Product Data Analyst", "Data Analyst Intern",
        ],
        "tech": [
            "SQL", "Python", "R", "pandas", "NumPy", "dbt", "Airflow", "Spark",
            "PySpark", "Snowflake", "BigQuery", "Redshift", "Databricks",
            "Tableau", "Looker", "Power BI", "Metabase",
        ],
        "topics": [
            "ETL pipelines", "ELT workflows", "data pipeline design",
            "data warehouse modelling", "star schema design", "A/B testing",
            "experimentation", "statistical significance", "cohort analysis",
            "funnel analysis", "data quality", "dashboard design",
            "SQL query optimisation", "metric definition",
        ],
        "formats": [
            "SQL screen", "take-home analysis", "case study presentation",
            "dashboard exercise", "analytics case interview", "data modelling round",
        ],
    },
    "machine_learning": {
        "roles": [
            "Machine Learning Engineer", "ML Engineer", "Research Scientist",
            "Applied Scientist", "AI Engineer", "Deep Learning Engineer",
            "MLOps Engineer", "ML Intern", "Research Intern",
            "Senior Machine Learning Engineer", "Junior ML Engineer",
            "Machine Learning Intern",
        ],
        "tech": [
            "PyTorch", "TensorFlow", "scikit-learn", "Hugging Face", "transformers",
            "Keras", "JAX", "LangChain", "CUDA", "SageMaker", "Vertex AI",
            "MLflow", "Weights & Biases", "vector databases", "Python",
        ],
        "topics": [
            "model training", "fine-tuning", "embeddings", "LLM applications",
            "transformer architecture", "neural networks", "feature engineering",
            "model deployment", "inference latency", "MLOps practice",
            "model evaluation", "precision and recall", "overfitting",
            "retrieval augmented generation", "prompt engineering", "GPU training",
        ],
        "formats": [
            "ML system design", "coding plus ML theory round", "paper discussion",
            "take-home modelling task", "applied ML case study",
        ],
    },
    "devops": {
        "roles": [
            "DevOps Engineer", "Site Reliability Engineer", "SRE",
            "Platform Engineer", "Infrastructure Engineer", "Cloud Engineer",
            "Systems Engineer", "Build Engineer", "DevOps Intern",
            "Senior Site Reliability Engineer", "Junior DevOps Engineer",
            "Platform Reliability Engineer",
        ],
        "tech": [
            "AWS", "GCP", "Azure", "EC2", "S3", "Lambda", "EKS", "GKE", "Docker",
            "Kubernetes", "k8s", "Terraform", "Ansible", "Helm", "Jenkins",
            "GitHub Actions", "GitLab CI", "ArgoCD", "Prometheus", "Grafana",
            "Datadog", "PagerDuty", "Linux", "bash", "Python",
        ],
        "topics": [
            "CI/CD pipelines", "infrastructure as code", "IaC practice",
            "observability", "monitoring and alerting", "incident response",
            "on-call rotation", "SLOs and SLIs", "uptime targets",
            "blue-green deployment", "autoscaling", "container orchestration",
            "cost optimisation", "disaster recovery",
        ],
        "formats": [
            "infrastructure troubleshooting exercise", "systems design round",
            "incident scenario walkthrough", "debugging exercise",
            "infrastructure take-home",
        ],
    },
    "security": {
        "roles": [
            "Security Engineer", "Application Security Engineer", "AppSec Engineer",
            "Security Analyst", "Penetration Tester", "Cloud Security Engineer",
            "SOC Analyst", "Security Intern", "Senior Security Engineer",
            "Product Security Engineer", "Offensive Security Engineer",
            "Information Security Analyst",
        ],
        "tech": [
            "Burp Suite", "Metasploit", "Nmap", "Wireshark", "Splunk",
            "SIEM tooling", "SAST tools", "DAST tools", "OWASP ZAP",
            "OSCP", "CISSP", "Security+", "CEH",
        ],
        "topics": [
            "penetration testing", "vulnerability assessment", "threat modelling",
            "the OWASP Top 10", "SQL injection", "XSS", "CSRF", "zero trust",
            "IAM policy", "encryption at rest", "TLS configuration",
            "incident response", "CVE triage", "secure code review",
            "red team exercises", "blue team operations", "security hardening",
        ],
        "formats": [
            "security assessment exercise", "CTF-style challenge",
            "threat modelling discussion", "secure code review round",
            "security take-home",
        ],
    },
    "qa": {
        "roles": [
            "QA Engineer", "QA Automation Engineer", "SDET",
            "Software Development Engineer in Test", "Test Engineer",
            "Quality Engineer", "Automation Engineer", "QA Analyst", "QA Intern",
            "Senior QA Engineer", "Junior Test Engineer", "Test Automation Engineer",
        ],
        "tech": [
            "Selenium", "Cypress", "Playwright", "Appium", "JUnit", "TestNG",
            "pytest", "Jest", "Postman", "JMeter", "Cucumber", "Robot Framework",
        ],
        "topics": [
            "test automation", "regression testing", "test plan authoring",
            "test case design", "end-to-end testing", "unit and integration tests",
            "load testing", "performance testing", "bug triage",
            "defect tracking in Jira", "test coverage", "exploratory testing",
            "manual testing", "CI test pipelines",
        ],
        "formats": [
            "test-design exercise", "automation take-home", "bug-finding exercise",
            "test strategy discussion", "live test-writing round",
        ],
    },
    "product_management": {
        "roles": [
            "Product Manager", "Associate Product Manager", "APM",
            "Technical Product Manager", "Senior Product Manager",
            "Group Product Manager", "Product Owner", "PM Intern",
            "Product Management Intern", "Junior Product Manager",
            "Platform Product Manager",
        ],
        "tech": [
            "Jira", "Confluence", "Amplitude", "Mixpanel", "Productboard",
            "Figma", "Looker", "Linear", "Notion",
        ],
        "topics": [
            "roadmap planning", "product strategy", "user research",
            "stakeholder management", "prioritisation frameworks", "RICE scoring",
            "OKRs", "KPI definition", "backlog grooming", "writing user stories",
            "PRD authoring", "product requirements", "go-to-market planning",
            "A/B testing", "north star metrics", "product discovery",
            "MVP scoping", "cross-functional collaboration",
        ],
        "formats": [
            "product sense interview", "product case study", "execution interview",
            "estimation question", "product design exercise", "metrics deep dive",
        ],
    },
    "design": {
        "roles": [
            "Product Designer", "UX Designer", "UI Designer", "UX Researcher",
            "Interaction Designer", "Visual Designer", "Senior Product Designer",
            "Design Intern", "Junior Designer", "UI/UX Designer",
            "Product Design Intern", "Senior UX Designer",
        ],
        "tech": [
            "Figma", "Sketch", "Adobe XD", "Framer", "Miro", "InVision",
            "Photoshop", "Illustrator", "Principle", "Figma prototypes",
        ],
        "topics": [
            "portfolio work", "design system contributions", "wireframing",
            "prototyping", "user research", "usability testing", "user flows",
            "information architecture", "accessibility", "WCAG standards",
            "design critique", "visual hierarchy", "typography",
            "interaction design", "end-to-end design process",
        ],
        "formats": [
            "portfolio presentation", "design exercise", "whiteboard challenge",
            "app critique round", "design take-home", "portfolio review",
        ],
    },
}


---
# TEMPLATES — part A
# interview_invitation, scheduling, rejection
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TEMPLATES — part A: interview_invitation, scheduling, rejection
# ══════════════════════════════════════════════════════════════════════════════
# A template is a FAMILY and the family is the grouping unit (template_id).
# Its subject variants and logistics paragraphs are OWNED by it and never shared
# with another template — subject lines are strongly category-determining, so a
# shared one would appear on both sides of the split and become memorisable.
# Greetings, sign-offs and signature blocks carry no label signal and ARE shared.
#
# Placeholders: {name} {recruiter} {role} {company} {tech} {tech2} {topic}
# {topic2} {format} {seniority} {sender_title} {interviewer} {platform}
# {duration} {date} {date2} {date3} {time} {time2} {time3} {location} {process}

templates_a = {

# ────────────────────────────────────────────────────────────────────────────
"interview_invitation": [
    {"subjects": ["Interview invitation - {role} at {company}",
                  "{company} | Interview for {role}"],
     "body": "Thanks for applying to the {role} role at {company}. The team reviewed your application and would like to move you forward to a {format}.\n\nYou'd be speaking with {interviewer}, and we'll focus mainly on {topic}.",
     "logistics": ["The session runs {duration} on {platform}. Are you free on {date} at {time}?",
                   "We're looking at {date} at {time}. Let me know if that works or if another day suits you better."]},

    {"subjects": ["Next steps on your {role} application",
                  "Moving forward - {role}"],
     "body": "I'm pleased to let you know we'd like to invite you to interview for the {role} position. Your experience with {tech} stood out to the team.\n\nThe {format} will cover {topic} and give you a chance to ask about how we work.",
     "logistics": ["It should take {duration}. Would {date} at {time} suit you?",
                   "Please let me know your availability over the next two weeks and I'll get it in the diary."]},

    {"subjects": ["We'd like to speak with you about the {role} role"],
     "body": "Your application for the {seniority} caught our attention, particularly your background in {topic}.\n\nWe'd like to set up a {format} as the first step. It's an informal conversation about your experience and what you're looking for.",
     "logistics": ["Roughly {duration} over {platform}. What does your availability look like next week?"]},

    {"subjects": ["{company} interview - {role}",
                  "Invitation to interview at {company}"],
     "body": "Following your application to {company}, the hiring team would like to invite you to a {format} for the {role} position.\n\nWe'll spend most of the time on {topic}, and there'll be room at the end for your questions.",
     "logistics": ["Planned for {duration} with {interviewer}, on {platform}.",
                   "The interview will be held at {location}, or remotely if you prefer."]},

    {"subjects": ["Interview request - {role}"],
     "body": "I lead recruiting for the engineering team at {company} and I've been asked to reach out about your application for the {role} opening.\n\nWe'd like to schedule a {format}. Familiarity with {tech} and {tech2} will be useful, though we're more interested in how you approach problems than specific tools.",
     "logistics": ["{duration}, on {platform}. I have {date} at {time} or {date2} at {time2} open."]},

    {"subjects": ["Your application to {company} - interview invitation"],
     "body": "Good news. After reviewing your CV for the {role} position, we'd like to progress to interview.\n\nThe first stage is a {format} focused on {topic}. If that goes well there would be a further round with {interviewer}.",
     "logistics": ["Expect around {duration}. Could you share a few times that work for you?"]},

    {"subjects": ["Invitation: {role} interview at {company}",
                  "{role} - interview stage"],
     "body": "Thank you for your interest in the {role} role. We were impressed with your application and would like to invite you to the next stage.\n\nThis is a {format}. It covers {topic} and {topic2}. No preparation is strictly required, but reviewing your recent projects would help.",
     "logistics": ["{duration} on {platform}. Does {date} at {time} work?"]},

    {"subjects": ["Let's set up an interview - {role}"],
     "body": "I wanted to reach out about the {role} position you applied for. The team liked what they saw and would like to talk.\n\nWe'd start with a {format}. Your work with {tech} is directly relevant to what the team is building.",
     "logistics": ["Fairly informal, {duration}. Let me know a couple of slots that suit you.",
                   "I'll send a calendar invite once you confirm a time."]},

    {"subjects": ["Interview for the {seniority} at {company}"],
     "body": "Thanks for putting your name forward for the {seniority} at {company}. We'd like to invite you to interview.\n\nThe process is two rounds: a {format} first, then a conversation with {interviewer}. The first round focuses on {topic}.",
     "logistics": ["Round one is {duration} on {platform}. What's your availability like?"]},

    {"subjects": ["{company} - next stage for your application"],
     "body": "I'm writing about your application for the {role} role. We'd like to take things further and invite you to a {format}.\n\nThe team works largely with {tech}, and we'll talk through {topic} during the session.",
     "logistics": ["Please let me know if {date} at {time} is convenient, or suggest an alternative."]},

    {"subjects": ["Interview invitation from {company}"],
     "body": "Your profile for the {role} opening was passed to me by the hiring manager, who'd like to meet you.\n\nWe'd like to arrange a {format}. It's a chance for both sides to see whether there's a fit before going further.",
     "logistics": ["{duration}, {platform}. I have several slots on {date} and {date2}."]},

    {"subjects": ["Interviewing for {role} - scheduling",
                  "{role} at {company} - let's talk"],
     "body": "Thank you for applying. We'd like to invite you to interview for the {role} position.\n\nThe {format} will be led by {interviewer} and will cover {topic}. We may also touch on your experience with {tech2}.",
     "logistics": ["Allow {duration}. Would either {date} at {time} or {date2} at {time2} suit?"]},

    {"subjects": ["Great news about your {role} application"],
     "body": "I've got good news. The hiring team reviewed your application for the {role} position and would like to move to interview.\n\nWe run a {format} at this stage. You'll be asked about {topic} and about a project you're proud of.",
     "logistics": ["It's scheduled for {duration}. Send me two or three times and I'll confirm."]},

    {"subjects": ["Invitation to interview - {company}"],
     "body": "On behalf of the team at {company}, I'd like to invite you to interview for the {role} role.\n\nWe'll run a {format} covering {topic}. If you'd like to prepare, having an example of your {tech} work ready would be useful.",
     "logistics": ["Held on {platform}, {duration}.",
                   "The interview is at {location}. Parking is available if you're driving."]},

    {"subjects": ["Your {role} application - interview"],
     "body": "Thanks for your patience while we worked through applications. We'd like to invite you to interview for the {role} role.\n\nThe format is a {format}, roughly split between {topic} and a discussion of your background.",
     "logistics": ["We're looking at the week of {date}. Which days work best?"]},

    {"subjects": ["{company} would like to interview you"],
     "body": "After reviewing candidates for the {role} position, we'd like to invite you to the interview stage.\n\nThe first conversation is a {format} with {interviewer}. Topics will include {topic} and {topic2}.",
     "logistics": ["{duration} on {platform}. Let me know what suits and I'll send the invite."]},

    {"subjects": ["Referral follow-up - {role} interview"],
     "body": "You were referred to us for the {role} opening, and having looked at your background I can see why.\n\nI'd like to arrange a {format}. Given the team's work with {tech}, your experience looks like a strong match.",
     "logistics": ["Around {duration}. Are you free on {date}?"]},

    {"subjects": ["Interview - {role} - please confirm availability"],
     "body": "We'd like to move ahead with your application for the {role} position and invite you to a {format}.\n\nThe session focuses on {topic}. Please note this is the {process} stage; there would be one further round after this.",
     "logistics": ["Duration {duration}, held over {platform}. Please confirm {date} at {time}."]},

    {"subjects": ["Speaking with you about {role} at {company}"],
     "body": "I hope you're well. I'm reaching out regarding your application for the {role} position at {company}.\n\nThe team would like to invite you to a {format}. We'll discuss {topic} and how you'd approach the kind of problems the team handles day to day.",
     "logistics": ["Please share your availability for the coming fortnight."]},

    {"subjects": ["Invitation - {seniority} interview"],
     "body": "Thanks for applying for the {seniority} at {company}. We'd like to invite you to interview.\n\nOur process starts with a {format}. It's mostly about {topic}, with some time on what you'd want from the role.",
     "logistics": ["{duration} with {interviewer} on {platform}. Does {date} at {time} work for you?"]},

    {"subjects": ["{role} - moving to interview stage"],
     "body": "I'm delighted to say your application for the {role} role has progressed. The team would like to interview you.\n\nWe run a {format} first. Expect questions on {topic}, and bring anything you'd like to show from your recent work with {tech}.",
     "logistics": ["Roughly {duration}. I'm holding {date} at {time} provisionally - let me know."]},

    {"subjects": ["Interview opportunity at {company}"],
     "body": "Your application for the {role} position has been shortlisted and we'd like to arrange an interview.\n\nThis stage is a {format} with {interviewer}, covering {topic}. It's a two-way conversation, so please come with questions.",
     "logistics": ["We can do {date} at {time}, {date2} at {time2}, or find something else that suits."]},
],

# ────────────────────────────────────────────────────────────────────────────
"scheduling": [
    {"subjects": ["Scheduling your {role} interview",
                  "Finding a time - {role} at {company}"],
     "body": "Following up to get a time in the calendar for your {role} interview.\n\nThe session is a {format} and will cover {topic}.",
     "logistics": ["Could you pick whichever of these suits you: {date} at {time}, {date2} at {time2}, or {date3} at {time3}?",
                   "Please let me know two or three windows that work and I'll confirm."]},

    {"subjects": ["Rescheduling - {role} interview",
                  "Need to move your interview"],
     "body": "Apologies, something has come up on our side and we need to move your {role} interview.\n\nEverything else stays the same - it's still the {format} with {interviewer}, covering {topic}.",
     "logistics": ["Could we do {date} at {time} or {date2} at {time2} instead? Sorry for the inconvenience."]},

    {"subjects": ["Confirming your interview time"],
     "body": "Just confirming the details for your {role} interview.\n\nIt's a {format} focused on {topic}, with {interviewer}.",
     "logistics": ["{date} at {time}, {duration}, on {platform}. Reply to confirm and I'll send the invite.",
                   "{date} at {time} at {location}. Please ask for me at reception."]},

    {"subjects": ["Your availability for the next round"],
     "body": "Thanks for your time so far. The team would like to bring you back for the next stage of the {role} process.\n\nThis round is a {format} covering {topic2} in more depth than last time.",
     "logistics": ["What does your calendar look like over the next two weeks?"]},

    {"subjects": ["Interview slots for {role}"],
     "body": "I've been asked to arrange your {format} for the {role} position.\n\nThe interviewer will be {interviewer} and the focus is {topic}.",
     "logistics": ["I have these open: {date} at {time}, {date2} at {time2}, {date3} at {time3}. Which would you prefer?"]},

    {"subjects": ["Calendar invite for your interview"],
     "body": "Getting your {role} interview booked in. The session covers {topic} and runs as a {format}.",
     "logistics": ["I've pencilled in {date} at {time}. Let me know if that clashes and I'll find another slot.",
                   "Sending a calendar invite for {date} at {time} on {platform}. Accept when you can."]},

    {"subjects": ["Time change - {company} interview"],
     "body": "A quick note about your upcoming {role} interview. {interviewer} has a conflict at the original time, so we need to shift things.\n\nThe content is unchanged - still a {format} on {topic}.",
     "logistics": ["Would {date} at {time} work instead? If not, send me a few alternatives."]},

    {"subjects": ["Booking your {format}"],
     "body": "Time to get your {format} in the diary for the {role} role.\n\nWe'll be looking at {topic}, and there'll be a short discussion of your experience with {tech}.",
     "logistics": ["It takes {duration}. Please reply with your preferred times.",
                   "Available slots are on {date} and {date2}. Let me know which day suits."]},

    {"subjects": ["Second round scheduling - {role}"],
     "body": "Good news - the team would like to progress you to the second round for the {role} position.\n\nThis is a longer session, a {format}, and goes deeper into {topic}.",
     "logistics": ["Allow {duration}. Which of {date}, {date2} or {date3} works best?"]},

    {"subjects": ["Sorting a time for your {role} conversation"],
     "body": "Trying to find a time for your conversation about the {role} opening.\n\nIt'll be a {format} with {interviewer}, mainly about {topic}.",
     "logistics": ["My calendar is fairly open next week. What suits you?"]},

    {"subjects": ["Interview logistics - {role} at {company}"],
     "body": "Here are the details for your upcoming {role} interview so you can plan.\n\nThe session is a {format}. You'll meet {interviewer} and cover {topic} plus some questions on {tech}.",
     "logistics": ["{duration} on {platform}. Can you confirm {date} at {time}?",
                   "We're at {location}. Allow a little extra time to get through the building."]},

    {"subjects": ["Can we move your interview?"],
     "body": "Unfortunately I need to ask about moving your {role} interview - {interviewer} has been pulled into something unavoidable.\n\nThe {format} itself is unchanged and still focuses on {topic}.",
     "logistics": ["I can offer {date} at {time} or {date2} at {time2}. Apologies for the disruption."]},

    {"subjects": ["Availability check - {role}"],
     "body": "Before I send anything formal, I wanted to check your availability for the {role} {format}.\n\nWe'd cover {topic} and give you time to ask about the team.",
     "logistics": ["Are mornings or afternoons generally better for you?"]},

    {"subjects": ["Final round scheduling"],
     "body": "You've reached the final stage for the {role} position. Congratulations on getting this far.\n\nThe last round is a {format} with {interviewer}, covering {topic2} and a broader conversation about the role.",
     "logistics": ["This one runs {duration}. Which day next week works?"]},

    {"subjects": ["Interview confirmed - {role}"],
     "body": "Confirming your {role} interview at {company}.\n\nFormat: {format}. Focus: {topic}. Interviewer: {interviewer}.",
     "logistics": ["{date} at {time}, {duration}, {platform}. Everything's in the calendar invite.",
                   "{date} at {time} at {location}. Let me know if you need anything beforehand."]},

    {"subjects": ["Picking a slot for your {role} interview"],
     "body": "We're ready to schedule your interview for the {role} role.\n\nIt's a {format} and will focus on {topic}. Some familiarity with {tech} would help but isn't essential.",
     "logistics": ["Options are {date} at {time}, {date2} at {time2}, or {date3} at {time3}."]},

    {"subjects": ["Following up on interview times"],
     "body": "Just following up as I haven't heard back about scheduling your {role} interview.\n\nThe {format} is still open and the team is keen to speak with you about {topic}.",
     "logistics": ["Do any of the times I sent still work, or shall I look at the following week?"]},

    {"subjects": ["Your interview with {interviewer}"],
     "body": "I'm arranging your session with {interviewer} for the {role} position.\n\nIt's a {format} and will centre on {topic}. Expect some discussion of {tech2} as well.",
     "logistics": ["{duration} on {platform}. Please confirm a time that works."]},

    {"subjects": ["Scheduling - {seniority}"],
     "body": "Time to arrange your interview for the {seniority} at {company}.\n\nThe {format} is the main assessment stage and covers {topic}.",
     "logistics": ["We have availability on {date} and {date2}. Which is better for you?"]},

    {"subjects": ["Interview reminder and details"],
     "body": "A reminder about your {role} interview coming up, plus a few details.\n\nYou'll be doing a {format} with {interviewer}. The focus is {topic}. Have an example of your {tech} work to hand.",
     "logistics": ["{date} at {time}. Please join {platform} a few minutes early to check your setup."]},

    {"subjects": ["Shifting your interview forward"],
     "body": "Would you be open to bringing your {role} interview forward? A slot has opened up and the team would like to move quickly.\n\nSame {format}, same focus on {topic}.",
     "logistics": ["We could do {date} at {time} instead of the original slot. No problem if not."]},

    {"subjects": ["Panel interview scheduling - {role}"],
     "body": "The next stage for the {role} position is a panel session, and I need to line up several diaries.\n\nYou'd meet {interviewer} across a {format}, covering {topic} and {topic2}.",
     "logistics": ["Because it's a panel I need a bit of notice. Could you send your availability for the week of {date}?"]},
],

# ────────────────────────────────────────────────────────────────────────────
"rejection": [
    {"subjects": ["Your application to {company}",
                  "Update on your {role} application"],
     "body": "Thank you for taking the time to apply for the {role} position at {company}.\n\nAfter careful consideration we've decided to move forward with other candidates whose experience more closely matches what the team needs right now. This was a difficult decision - we had a very strong field.",
     "logistics": ["We'd genuinely welcome an application from you in future if something suitable opens up."]},

    {"subjects": ["{role} at {company} - decision"],
     "body": "I'm writing with an update on your application for the {role} role.\n\nUnfortunately we won't be taking things further on this occasion. The team was looking for deeper experience with {topic} than your background currently shows.",
     "logistics": ["I'll keep your details on file and get in touch if a closer match comes up."]},

    {"subjects": ["Thank you for interviewing with us"],
     "body": "Thank you for making the time to interview for the {role} position, and for the effort you put into the {format}.\n\nAfter discussing with the team we've decided not to proceed. The standard was high and it came down to some very fine margins.",
     "logistics": ["If it would be useful I'm happy to share some feedback from the panel - just let me know."]},

    {"subjects": ["Update from {company}"],
     "body": "Thanks for your interest in the {seniority} at {company} and for the time you invested in the process.\n\nWe've now completed our review and won't be moving ahead with your application. The role attracted a large number of applicants with directly relevant {topic} experience.",
     "logistics": []},

    {"subjects": ["Regarding your {role} application"],
     "body": "I wanted to come back to you personally rather than leave you waiting.\n\nWe've decided not to progress your application for the {role} position. Your background is strong, but the team has settled on a candidate whose {tech} experience lines up more directly with the immediate work.",
     "logistics": ["Please don't read this as a reflection on your ability. Do keep an eye on our careers page."]},

    {"subjects": ["Your application - {company}"],
     "body": "Thank you for applying for the {role} role.\n\nOn this occasion we've chosen to progress other applicants. We appreciate the time you spent on your application and wish you well with your search.",
     "logistics": []},

    {"subjects": ["Outcome of your interview"],
     "body": "Thank you for coming in for the {format} last week. The team enjoyed speaking with you.\n\nI'm sorry to say we won't be making an offer. The discussion on {topic} didn't go quite as deep as we needed for this particular role.",
     "logistics": ["You interviewed well and I'd encourage you to apply again once you've had more exposure to {topic2}."]},

    {"subjects": ["{company} - application update"],
     "body": "An update on the {role} position you applied for.\n\nWe've filled the role internally, so your application has been unsuccessful and we're closing the external process. I'm sorry we couldn't take it further - this wasn't about the quality of your application.",
     "logistics": ["I'd be glad to hear from you again when we next advertise."]},

    {"subjects": ["Thank you for your interest in {company}"],
     "body": "Thank you for your interest in the {role} opening.\n\nAfter reviewing all applications we've decided to move forward with other candidates. We had far more strong applicants than places.",
     "logistics": []},

    {"subjects": ["Following your final round"],
     "body": "Thank you for the time you gave us across all three rounds for the {role} position, and particularly for the work you put into the {format}.\n\nThis was a genuinely close decision, but we've offered the role to another candidate.",
     "logistics": ["You were a strong finalist. If another {role} opening comes up I will contact you directly."]},

    {"subjects": ["Your {role} application at {company}"],
     "body": "I'm writing to let you know we won't be progressing your application for the {role} position.\n\nThe team was specifically looking for someone with production experience in {tech}, and that gap was the deciding factor.",
     "logistics": []},

    {"subjects": ["Decision on the {seniority}"],
     "body": "Thanks for applying for the {seniority} at {company}.\n\nWe've completed our shortlisting and unfortunately your application wasn't successful this time. Competition for our early-careers roles is particularly high.",
     "logistics": ["Our next intake opens later in the year and you'd be very welcome to apply again."]},

    {"subjects": ["Not moving forward - {role}"],
     "body": "I'll be direct since I know waiting is the worst part: we're not moving forward with your application for the {role} role.\n\nThe team went with someone who has spent longer working on {topic}. That was the whole of it.",
     "logistics": ["Thank you for your interest in {company} and good luck with the rest of your search."]},

    {"subjects": ["Update following your {format}"],
     "body": "Thank you for completing the {format} for the {role} position.\n\nThe team reviewed your submission carefully. Unfortunately we've decided not to progress - the solution worked, but we were looking for more attention to {topic}.",
     "logistics": ["I'm happy to pass on the reviewer's detailed notes if you'd find that helpful."]},

    {"subjects": ["{company} hiring update"],
     "body": "I wanted to update you on the {role} process.\n\nWe've paused hiring for this position due to a change in headcount planning, so we won't be taking your application forward and are closing all open applications. This isn't a reflection on your candidacy at all.",
     "logistics": ["If the role reopens I'll come back to you first."]},

    {"subjects": ["Your recent application"],
     "body": "Thank you for applying to {company} for the {role} role.\n\nWe've reviewed your CV against the requirements and have decided not to take your application forward on this occasion.",
     "logistics": []},

    {"subjects": ["After careful consideration - {role}"],
     "body": "Thank you for interviewing for the {role} position and for your patience during our decision process.\n\nAfter careful consideration we've decided to proceed with another candidate. The panel was impressed by your approach to {topic} but felt the overall fit was closer elsewhere.",
     "logistics": ["I hope you'll consider us again in future."]},

    {"subjects": ["Unsuccessful this time - {role}"],
     "body": "I'm sorry to say your application for the {role} position has been unsuccessful.\n\nWe were looking for someone able to work independently on {topic} from day one, and on balance the team felt another candidate was better placed for that.",
     "logistics": []},

    {"subjects": ["Result of your {company} interview"],
     "body": "Thank you for meeting with {interviewer} last week to discuss the {role} role.\n\nI'm afraid we've decided not to make an offer. The feedback was positive on your {tech} knowledge, but the team needed stronger experience in {topic2}.",
     "logistics": ["Genuinely, this was close. Please do stay in touch."]},

    {"subjects": ["Closing your application - {company}"],
     "body": "This is to confirm that we've closed your application for the {role} position.\n\nWe received an exceptional response to this advert and were only able to progress a small number of candidates.",
     "logistics": ["Thank you for considering {company} and best of luck."]},

    {"subjects": ["Your candidacy for {role}"],
     "body": "Thank you for the conversation about the {role} opening.\n\nHaving discussed it as a team we've decided not to move forward. What we need right now is someone already deep in {topic}, and that's a narrow requirement rather than a judgement on your work.",
     "logistics": ["I'd be happy to keep you in mind for other openings at {company}."]},

    {"subjects": ["Application outcome"],
     "body": "Following our review of applications for the {role} position, I'm writing to let you know that we won't be progressing yours.\n\nWe appreciate the effort involved in applying and thank you for your interest in {company}.",
     "logistics": []},
],
}


---
# TEMPLATES — part B
# offer, recruiter_outreach, follow_up, application_received
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TEMPLATES — part B: offer, recruiter_outreach, follow_up, application_received
# ══════════════════════════════════════════════════════════════════════════════
# application_received is the new seventh category. It is deliberately written to
# be distinguishable from follow_up, which is why it needed new templates rather
# than a relabel: application_received is the AUTOMATED "we have your
# application, no action needed" acknowledgement, whereas follow_up is a human
# checking in mid-process. Previously no template expressed the former, so the
# classifier had no correct label available and put confirmations in 'rejection'.

templates_b = {

# ────────────────────────────────────────────────────────────────────────────
"offer": [
    {"subjects": ["Offer - {role} at {company}", "We'd like to offer you the {role} role"],
     "body": "I'm delighted to offer you the {role} position at {company}.\n\nThe team was unanimous after your {format} - your approach to {topic} was exactly what they were hoping for.",
     "logistics": ["The full written offer is attached. Please review and let me know if you have questions.",
                   "I'll send the formal paperwork today. We'd love an answer within a week if possible."]},

    {"subjects": ["Your offer from {company}"],
     "body": "Congratulations. Following your interviews we'd like to offer you the {role} position.\n\nThe package details are in the attached letter, including salary, equity and benefits.",
     "logistics": ["Happy to talk any of it through on a call - just say when.",
                   "The offer is open until the end of next week."]},

    {"subjects": ["Great news - {role}"],
     "body": "I have good news. {company} would like to offer you the {role} role.\n\nEveryone you met was impressed, particularly with how you handled the {topic} discussion.",
     "logistics": ["Details attached. Take the time you need, but do let me know if anything is unclear."]},

    {"subjects": ["Offer letter - {seniority}"],
     "body": "We'd like to offer you the {seniority} at {company}.\n\nYou'd be joining the team working primarily with {tech} and {tech2}, reporting to {interviewer}.",
     "logistics": ["The offer letter is attached with a proposed start date. Let me know if that timing works."]},

    {"subjects": ["Welcome to {company} - offer details"],
     "body": "It's my pleasure to extend an offer for the {role} position.\n\nThe hiring panel felt your experience with {topic} would make an immediate difference to what the team is building.",
     "logistics": ["Please find compensation and start date details attached. I'm around all week to discuss."]},

    {"subjects": ["{company} offer - {role}"],
     "body": "Following the {format}, I'm pleased to confirm that we would like to offer you the {role} role.\n\nThis is subject to the usual {process}, which is a formality in most cases.",
     "logistics": ["Formal documentation to follow. Do come back to me with any questions on the terms."]},

    {"subjects": ["Offer of employment"],
     "body": "On behalf of {company} I'm pleased to offer you the position of {role}.\n\nThe team is genuinely excited - your work on {topic} came up repeatedly in the debrief.",
     "logistics": ["Everything is in the attached letter. We'd appreciate a decision by {date} if you can."]},

    {"subjects": ["We'd like you to join us"],
     "body": "After a very competitive process, we'd like you to join {company} as our new {role}.\n\nYour depth on {tech} was the standout factor for the panel.",
     "logistics": ["Offer details attached. Let me know a good time to call and walk through it."]},

    {"subjects": ["Your {role} offer"],
     "body": "Congratulations on reaching the end of the process. We're delighted to offer you the {role} position at {company}.\n\nYou'd be joining a team of eight, working on {topic}.",
     "logistics": ["The written offer covers salary, holiday and the benefits package. Take a look and let me know."]},

    {"subjects": ["Offer - please review"],
     "body": "We would like to offer you the {role} role at {company}.\n\nI've put the key terms below and the full contract is attached for your review.",
     "logistics": ["I know this is a big decision. Please ask about anything at all before you decide."]},

    {"subjects": ["Delighted to make you an offer"],
     "body": "I'm delighted to be writing with an offer for the {role} position.\n\nThe feedback from your {format} was some of the strongest we've had this year, especially around {topic}.",
     "logistics": ["Contract attached. Start date is flexible if you need to give notice."]},

    {"subjects": ["{role} - offer confirmed"],
     "body": "This confirms our verbal offer for the {role} position at {company}.\n\nAs discussed, you'd be working with {tech} and joining the team led by {interviewer}.",
     "logistics": ["Written terms attached to match what we discussed on the phone."]},

    {"subjects": ["An offer for you - {company}"],
     "body": "We've completed our process for the {role} role and you're our first choice.\n\nI'd like to formally offer you the position. Everyone felt you'd fit well with how the team works.",
     "logistics": ["Package details attached. Happy to be flexible on the start date."]},

    {"subjects": ["Offer for the {seniority}"],
     "body": "Congratulations - we'd like to offer you the {seniority} at {company}.\n\nThe programme runs alongside the {topic} team, and you'd be paired with a mentor from day one.",
     "logistics": ["Please confirm acceptance by {date} so we can get your onboarding started."]},

    {"subjects": ["Job offer - {role} at {company}"],
     "body": "I'm pleased to confirm that {company} is offering you the {role} position.\n\nThis follows your {format} and the panel discussion afterwards, which was very positive.",
     "logistics": ["The offer is attached. Please read carefully and come back with questions."]},

    {"subjects": ["Offer and next steps"],
     "body": "We'd like to offer you the {role} role. Congratulations.\n\nThe team was particularly taken with your explanation of {topic} - it's exactly the thinking they need.",
     "logistics": ["Once you've accepted, the next steps are {process} and then onboarding paperwork."]},

    {"subjects": ["Revised offer - {role}"],
     "body": "Thank you for coming back to me about the terms. I've discussed it internally and we're able to improve the original offer for the {role} position.\n\nEverything else stays as it was.",
     "logistics": ["Revised letter attached. I hope this addresses what we discussed."]},

    {"subjects": ["You've got the job"],
     "body": "You've got it. We'd like you to join {company} as our {role}.\n\nThe whole panel backed the decision after your work on {topic} in the final round.",
     "logistics": ["Formal offer follows this email. Congratulations - the team is looking forward to it."]},

    {"subjects": ["Offer details for your review"],
     "body": "Further to your interviews for the {role} role, I'm pleased to make you a formal offer.\n\nYou'd be joining the team at {location} and working mainly with {tech}.",
     "logistics": ["All the details are attached. Let me know if you'd like to talk it through."]},

    {"subjects": ["Congratulations from {company}"],
     "body": "Congratulations. We'd like to offer you the {role} position.\n\nYour depth on {topic} and the way you communicated it made this a straightforward decision for the panel.",
     "logistics": ["Attached is the offer. Take your time, and shout if anything needs clarifying."]},

    {"subjects": ["Formal offer - {role}"],
     "body": "It gives me great pleasure to offer you the role of {role} at {company}.\n\nThis offer is conditional on {process}, which we'll begin once you accept.",
     "logistics": ["Please sign and return the attached letter at your convenience."]},

    {"subjects": ["Offer - {role} - {company}"],
     "body": "We're pleased to offer you the {role} position following your recent interviews.\n\nThe team is keen to get you started on the {topic} work as soon as you're available.",
     "logistics": ["Compensation and start date are in the attached document. Let me know your thoughts."]},
],

# ────────────────────────────────────────────────────────────────────────────
"recruiter_outreach": [
    {"subjects": ["{role} opportunity at {company}", "Are you open to new roles?"],
     "body": "I came across your profile and thought of an opening we have for a {role} at {company}.\n\nThe team works with {tech} and is building out its {topic} capability. Might be worth a conversation.",
     "logistics": ["Would you be open to a short call this week?"]},

    {"subjects": ["Interested in a {role} role?"],
     "body": "I'm a {sender_title} at {company} and I'm recruiting for a {role} position.\n\nYour background in {topic} looked like a genuine match, which is why I'm reaching out directly rather than through an advert.",
     "logistics": ["Happy to send the full job description if you're curious - no pressure either way."]},

    {"subjects": ["Opportunity - {seniority} at {company}"],
     "body": "I hope you don't mind the cold email. We're hiring for a {seniority} at {company} and your profile stood out.\n\nThe role involves a lot of {topic} work and the stack is mostly {tech}.",
     "logistics": ["Let me know if you'd like to hear more."]},

    {"subjects": ["Your profile - {role} opening"],
     "body": "I'm reaching out about a {role} vacancy on my team.\n\nWe need someone comfortable with {tech} who enjoys {topic}. From your profile that looks like a reasonable description of you.",
     "logistics": ["Would a 15-minute call make sense? I can work around your schedule."]},

    {"subjects": ["Hiring: {role}"],
     "body": "{company} is hiring a {role} and I wanted to get in touch before the role goes public.\n\nIt's a growing team and the work centres on {topic}.",
     "logistics": ["Interested in an informal chat?"]},

    {"subjects": ["Quick question about your next move"],
     "body": "Are you open to hearing about a {role} position?\n\nI'm recruiting for {company} and the team is looking for someone with solid {tech} experience to work on {topic}.",
     "logistics": ["If the timing is wrong I completely understand - just let me know."]},

    {"subjects": ["{company} is looking for a {role}"],
     "body": "I lead technical recruiting at {company} and we have a {role} opening I think you'd find interesting.\n\nThe team owns {topic} end to end and works primarily in {tech}.",
     "logistics": ["Do you have 20 minutes this week or next for a no-obligation conversation?"]},

    {"subjects": ["Reaching out about a role"],
     "body": "Apologies for the unsolicited message. I'm hiring a {role} at {company} and your experience with {tech2} caught my eye.\n\nThe position focuses on {topic}, with scope to shape how the team approaches it.",
     "logistics": ["Shall I send over the details?"]},

    {"subjects": ["A {role} role you might like"],
     "body": "I'm working on a {role} search for {company} and thought of you.\n\nThe role is heavy on {topic}. They're flexible on years of experience and more interested in how you think.",
     "logistics": ["Worth a chat? I'm free most afternoons."]},

    {"subjects": ["Exploring new opportunities?"],
     "body": "I wanted to see whether you're open to exploring new roles at the moment.\n\nWe're hiring a {role} at {company}. The work is largely {topic}, using {tech}.",
     "logistics": ["Even if now isn't right, I'd be glad to stay in touch for future openings."]},

    {"subjects": ["{role} - {company} - would you be interested?"],
     "body": "I'm recruiting for a {role} at {company} and your profile matched several of the things the hiring manager asked for.\n\nParticularly your work on {topic}.",
     "logistics": ["Can I send you the spec?"]},

    {"subjects": ["An opening on our team"],
     "body": "There's an opening on my team for a {role} and I'm reaching out to a handful of people directly.\n\nThe day-to-day is {topic}, and we use {tech} across most of the codebase.",
     "logistics": ["Let me know if you'd like to talk. Happy to answer questions by email too."]},

    {"subjects": ["Considering a change?"],
     "body": "I'm a {sender_title} at {company}. We're growing the team and looking for a {role}.\n\nThe role suits someone who enjoys {topic} and wants more ownership than a large company typically offers.",
     "logistics": ["Would you be open to a short introductory call?"]},

    {"subjects": ["{seniority} at {company}"],
     "body": "We've just opened a {seniority} at {company} and I thought it might be relevant to you.\n\nIt's a good route in for someone with {topic} exposure looking to build depth.",
     "logistics": ["Applications open next month but I'm happy to talk beforehand."]},

    {"subjects": ["Sourcing for a {role} position"],
     "body": "I'm sourcing candidates for a {role} role at {company}.\n\nThe team is small and works on {topic}. Experience with {tech} is preferred but they'll consider adjacent backgrounds.",
     "logistics": ["Interested? I can share more detail on the team and comp range."]},

    {"subjects": ["Thought of you for this {role} role"],
     "body": "I saw your work and immediately thought of a {role} opening I'm handling.\n\n{company} needs someone who can own {topic}. It's a role with real scope.",
     "logistics": ["Happy to jump on a call whenever suits, or keep it to email if you prefer."]},

    {"subjects": ["Hiring for {topic} work"],
     "body": "We're expanding the team at {company} and hiring a {role}.\n\nThe focus is {topic}, and the existing stack is {tech} with some {tech2}.",
     "logistics": ["Let me know if you'd like the full description."]},

    {"subjects": ["Would this {role} role suit you?"],
     "body": "I'm reaching out about a {role} position at {company}.\n\nI'll be upfront: they want someone strong on {topic}. If that's not your area, no problem, but your profile suggested it might be.",
     "logistics": ["Reply if you'd like to know more."]},

    {"subjects": ["New {role} opening"],
     "body": "A {role} role has just opened at {company} and I'm contacting a small number of people before it's advertised.\n\nThe team works on {topic}. It's a permanent position with hybrid working.",
     "logistics": ["Would you like me to send the details over?"]},

    {"subjects": ["Introduction - {sender_title} at {company}"],
     "body": "I'm the {sender_title} at {company}. We're recruiting a {role} and I wanted to introduce myself.\n\nThe role centres on {topic} and works closely with the {tech} platform team.",
     "logistics": ["No pressure at all - just let me know if you'd like to explore it."]},

    {"subjects": ["Is your inbox open to recruiters?"],
     "body": "I know these messages can be unwelcome, so I'll keep it short. We're hiring a {role} at {company}.\n\nThe work is {topic}, mostly in {tech}. The team is well established and the manager is excellent.",
     "logistics": ["One reply either way and I won't chase. Thanks for reading."]},

    {"subjects": ["{company} - {role} - initial contact"],
     "body": "I'm getting in touch about a {role} vacancy at {company}.\n\nYour experience looks well suited, particularly around {topic}. The team is currently four people and growing to seven.",
     "logistics": ["Can we find 20 minutes to talk this week?"]},
],

# ────────────────────────────────────────────────────────────────────────────
"follow_up": [
    {"subjects": ["Checking in on your {role} application",
                  "Update on where we are"],
     "body": "I wanted to check in and let you know where things stand with your {role} application.\n\nWe're still working through interviews and expect to have a decision by the end of next week. Apologies for the wait.",
     "logistics": ["Thanks for your patience - I'll come back to you as soon as I know more."]},

    {"subjects": ["Quick update - {role}"],
     "body": "Just a short note so you're not left wondering.\n\nYour application for the {role} position is still active. The hiring manager has been travelling, which has slowed the process down.",
     "logistics": []},

    {"subjects": ["Following up after your interview"],
     "body": "Thank you again for the {format} last week. I wanted to follow up.\n\nThe panel is still gathering feedback, particularly on the {topic} section. I should have news for you shortly.",
     "logistics": ["Do shout if you have questions in the meantime."]},

    {"subjects": ["Still with us - {role} process"],
     "body": "I know it's been quiet, so here's an update.\n\nYou're still in the running for the {role} position. We're waiting on one more candidate to complete their {format} before the team makes a decision.",
     "logistics": ["I expect to be able to tell you more by {date}."]},

    {"subjects": ["Apologies for the delay"],
     "body": "Apologies for the silence on your {role} application.\n\nWe've had some internal reorganisation which has held things up. Your application is still under consideration and nothing has been decided.",
     "logistics": ["I'll chase internally and come back to you this week."]},

    {"subjects": ["Where things stand - {company}"],
     "body": "An update on your application for the {role} role.\n\nWe've completed first-round interviews and are shortlisting for the next stage. You're on the shortlist we're discussing.",
     "logistics": ["Nothing needed from you right now - I'll be in touch."]},

    {"subjects": ["Any questions since we spoke?"],
     "body": "It's been a couple of weeks since your {format} for the {role} position and I wanted to reconnect.\n\nThe process is taking longer than planned because we've extended the search slightly.",
     "logistics": ["If you've got other offers in play, tell me and I'll try to speed things up."]},

    {"subjects": ["Checking you're still interested"],
     "body": "I wanted to check whether you're still interested in the {role} position at {company}.\n\nOur process has run longer than we'd like and I'd rather ask than assume.",
     "logistics": ["A quick yes or no is all I need."]},

    {"subjects": ["Update on the {role} search"],
     "body": "A quick update: we've had to pause the {role} search briefly while budget is confirmed for the next quarter.\n\nYour application remains active and we haven't rejected anyone.",
     "logistics": ["I'll write again as soon as I have a firm timeline."]},

    {"subjects": ["Following up on your application"],
     "body": "Following up on your {role} application from a few weeks ago.\n\nWe're still reviewing. The volume of applications for roles involving {topic} has been higher than expected.",
     "logistics": []},

    {"subjects": ["Progress update - {seniority}"],
     "body": "I wanted to let you know how the {seniority} process is progressing.\n\nWe've finished screening and are now arranging interviews. You should hear from the scheduling team shortly.",
     "logistics": ["No action needed from you at this stage."]},

    {"subjects": ["Since we last spoke"],
     "body": "Since we last spoke about the {role} role, a couple of things have changed on our side.\n\nThe team has grown and the position now leans more towards {topic2} than originally described. I wanted to flag that in case it changes your view.",
     "logistics": ["Happy to talk it through if useful."]},

    {"subjects": ["Your application is progressing"],
     "body": "Good news, of a sort: your application for the {role} position has cleared the first review and moved to the hiring manager.\n\nThey'll assess against the {topic} requirements and come back to us.",
     "logistics": ["Expect to hear something within a week or so."]},

    {"subjects": ["Still reviewing - thanks for waiting"],
     "body": "Thanks for your patience. We're still reviewing candidates for the {role} position.\n\nThe {format} responses have taken longer to assess than we anticipated.",
     "logistics": ["I'll be in touch by the end of the week either way."]},

    {"subjects": ["Reconnecting about {role}"],
     "body": "We spoke a while ago about the {role} opening at {company}. I wanted to reconnect.\n\nThe role is still open and we've adjusted the requirements slightly - less emphasis on {tech}, more on {topic}.",
     "logistics": ["Would you like to pick the conversation back up?"]},

    {"subjects": ["Interview feedback coming"],
     "body": "I haven't forgotten you. Feedback from your {role} interview is still being collated.\n\n{interviewer} has been out of office which has delayed the debrief.",
     "logistics": ["I should have something concrete for you by {date}."]},

    {"subjects": ["Checking in - {company}"],
     "body": "Just checking in on your {role} application.\n\nThere's no decision yet. We're interviewing two more candidates this week and then the team will compare notes.",
     "logistics": []},

    {"subjects": ["A note on timing"],
     "body": "I wanted to be transparent about timing on the {role} role.\n\nRealistically we won't make a decision for another two weeks. I know that's frustrating and I'm sorry.",
     "logistics": ["If your situation changes, please tell me and I'll do what I can."]},

    {"subjects": ["Following up on our conversation"],
     "body": "Following our conversation about the {role} position, I wanted to see if you had any further questions.\n\nWe're keen to keep you engaged while the process completes.",
     "logistics": ["Happy to arrange an informal chat with someone on the team if that would help."]},

    {"subjects": ["{role} - status update"],
     "body": "Status update on the {role} position: we're at the final stage with a small number of candidates, yourself included.\n\nThe last {format} is scheduled for this week.",
     "logistics": ["I'll come back to everyone once that's done."]},

    {"subjects": ["Sorry for the wait"],
     "body": "I owe you an update on your {role} application. Sorry it's been so long.\n\nThe hiring manager changed partway through the process, which reset some of the review. Your application is still live.",
     "logistics": ["I'll make sure you hear from us this week."]},

    {"subjects": ["Touching base"],
     "body": "Touching base about the {role} opportunity at {company}.\n\nWe're moving more slowly than planned but the role is very much still open. Your {topic} experience is still of interest to the team.",
     "logistics": ["Let me know if anything has changed on your side."]},
],

# ────────────────────────────────────────────────────────────────────────────
"application_received": [
    {"subjects": ["We received your application - {company}",
                  "Application received: {role}"],
     "body": "Thank you for applying to the {role} position at {company}.\n\nThis email confirms that we have received your application. Our hiring team reviews every submission and will be in touch if your background matches what we're looking for.",
     "logistics": ["No action is needed from you at this time.",
                   "Please do not reply to this message - it is sent from an unmonitored address."]},

    {"subjects": ["Your application to {company} has been received"],
     "body": "We've received your application for the {role} role.\n\nOur team will review it against the requirements over the coming days. If there's a match, a member of the recruiting team will contact you to arrange a {format}.",
     "logistics": ["You can check the status of your application at any time by logging into our careers portal."]},

    {"subjects": ["Thanks for applying - {role}"],
     "body": "Thanks for your interest in the {role} position at {company}. Your application is now in our system.\n\nWe typically review applications within two weeks of the closing date.",
     "logistics": ["We'll be in touch either way once the review is complete."]},

    {"subjects": ["Application confirmation"],
     "body": "This is an automated confirmation that your application for the {seniority} has been received.\n\nApplications are processed through our {process}, and successful candidates are contacted directly by a recruiter.",
     "logistics": ["Please retain this email for your records. Reference: your application to {company}."]},

    {"subjects": ["We have your application - {company}"],
     "body": "Thank you for taking the time to apply for the {role} opening.\n\nWe have your CV and application form. The hiring manager will review candidates once the advert closes.",
     "logistics": ["If you are shortlisted we'll contact you within three weeks."]},

    {"subjects": ["Received: your {role} application"],
     "body": "We've received your application for the {role} position and wanted to acknowledge it straight away.\n\nGiven the volume of applications we receive for roles involving {topic}, please allow us some time to respond.",
     "logistics": []},

    {"subjects": ["Application submitted successfully"],
     "body": "Your application for the {role} position at {company} has been submitted successfully.\n\nWhat happens next: our recruiting team screens applications, then shortlisted candidates are invited to a {format}.",
     "logistics": ["The whole process usually takes three to four weeks from the closing date."]},

    {"subjects": ["Thank you for your application"],
     "body": "Thank you for applying to {company}. We've received your application for the {role} role.\n\nEvery application is read by a person, not just filtered automatically, so it may take us a little longer to come back to you.",
     "logistics": ["We appreciate your patience."]},

    {"subjects": ["{company} - application acknowledgement"],
     "body": "This message confirms receipt of your application for the {role} position.\n\nYour details have been added to our {process} and will be reviewed by the hiring team.",
     "logistics": ["This is an automated message. Please do not reply."]},

    {"subjects": ["Your application is being processed"],
     "body": "We're writing to confirm that your application for the {seniority} at {company} has been received and is being processed.\n\nOur early careers team reviews all applications after the deadline rather than on a rolling basis.",
     "logistics": ["You should hear from us by the end of the month."]},

    {"subjects": ["Application received - next steps"],
     "body": "Thanks for applying for the {role} role. We have your application.\n\nIf your experience with {topic} matches what the team needs, we'll invite you to a {format} as the first stage.",
     "logistics": ["There's nothing you need to do right now."]},

    {"subjects": ["We got your application"],
     "body": "Just to let you know we've got your application for the {role} position at {company}.\n\nIt's with the hiring team now. We aim to give everyone an answer, even when it's a no.",
     "logistics": ["Thanks for your interest in {company}."]},

    {"subjects": ["Confirmation of application - {role}"],
     "body": "This confirms that {company} has received your application for the {role} position.\n\nApplications close on {date}, after which the review process begins.",
     "logistics": ["We'll email all applicants with an outcome once shortlisting is complete."]},

    {"subjects": ["Your submission has been received"],
     "body": "Your submission for the {role} opening has been received.\n\nWe review applications in the order received. Roles requiring {tech} experience tend to attract a high volume, so please bear with us.",
     "logistics": []},

    {"subjects": ["Application logged - {company}"],
     "body": "Your application for the {role} position has been logged in our system.\n\nA recruiter will screen it against the role requirements. If shortlisted, you'll be contacted to arrange an initial conversation.",
     "logistics": ["Typical response time is 10 to 15 working days."]},

    {"subjects": ["Thanks - we've received your CV"],
     "body": "Thanks for sending your CV for the {role} role at {company}.\n\nWe've added it to the shortlist pile for review. The hiring manager looks at every CV personally for this position.",
     "logistics": ["We'll get back to you as soon as we can."]},

    {"subjects": ["Application received for {seniority}"],
     "body": "Thank you for applying for the {seniority} at {company}.\n\nWe have your application. Our graduate recruitment process involves an initial screen followed by a {format}.",
     "logistics": ["Full details of the process are on our careers site."]},

    {"subjects": ["Acknowledging your application"],
     "body": "We are acknowledging receipt of your application for the {role} position.\n\nNo further action is required from you at this stage. Our team will assess your suitability and respond in due course.",
     "logistics": ["This is a system-generated acknowledgement."]},

    {"subjects": ["Your {role} application - received"],
     "body": "We've received your application for the {role} role and wanted to confirm it arrived safely.\n\nThe team will start reviewing next week once the advert closes.",
     "logistics": ["If you applied for more than one role you'll receive a separate confirmation for each."]},

    {"subjects": ["Application complete"],
     "body": "Your application for the {role} position at {company} is complete and has been received.\n\nYou'll be contacted by a member of the recruiting team if you're selected to move forward to a {format}.",
     "logistics": ["Thank you for considering {company} as your next step."]},

    {"subjects": ["Received your application for {role}"],
     "body": "Thank you - your application for the {role} position has reached us.\n\nWe know applying takes effort and we appreciate you choosing to apply to {company}.",
     "logistics": ["We'll be in touch once the review is complete. No need to follow up in the meantime."]},

    {"subjects": ["Confirmation - {company} careers"],
     "body": "This is confirmation that your application for the {role} opening has been received by {company}.\n\nOur process is: application review, recruiter screen, then a {format} with the team.",
     "logistics": ["You'll hear from us at each stage. Please do not reply to this address."]},
],
}


---
# URGENCY WEIGHTS
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# URGENCY WEIGHTS
# ══════════════════════════════════════════════════════════════════════════════
# Flattened so every category spans at least two levels with real mass — the
# marker sentence has to carry the within-category distinction or the head learns
# nothing and falls back to fitting names.
#
# The zeros are semantic, not cosmetic. An interview invitation, a scheduling
# email and an offer ALWAYS ask the reader to do something, so LOW is not a
# reachable label for them; a rejection and an automated acknowledgement never
# impose a deadline, so HIGH is not reachable for those. Category therefore still
# narrows the answer to two of three — real and unavoidable — but no longer
# determines it. Majority-per-category baseline falls 0.781 -> ~0.56.
urgency_weights = {
    "interview_invitation": [("high", 0.50), ("medium", 0.50)],
    "scheduling":           [("high", 0.45), ("medium", 0.55)],
    "offer":                [("high", 0.50), ("medium", 0.50)],
    "recruiter_outreach":   [("high", 0.10), ("medium", 0.50), ("low", 0.40)],
    "follow_up":            [("high", 0.15), ("medium", 0.40), ("low", 0.45)],
    "application_received": [("medium", 0.30), ("low", 0.70)],
    "rejection":            [("medium", 0.25), ("low", 0.75)],
}


def choose_weighted(options):
    labels  = [label  for label,  _ in options]
    weights = [weight for _, weight in options]
    return random.choices(labels, weights=weights, k=1)[0]


---
# ASSEMBLY
# Subject + greeting + body + tech context + logistics + courtesy
# + closing + signature
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ASSEMBLY
# ══════════════════════════════════════════════════════════════════════════════
def fill_slots(text, field_data, company):
    """
    Substitute every placeholder. Field vocabulary ({role}/{tech}/{topic}/
    {format}) comes from field_data; everything else comes from the shared pools,
    which is what keeps names and companies independent of the label.

    company is passed in rather than drawn here so that the subject, body and
    signature of one email all name the SAME company.

    {tech}/{tech2} and {topic}/{topic2} are drawn without replacement so a row
    never names the same technology twice.
    """
    techs  = random.sample(field_data["tech"],   min(2, len(field_data["tech"])))
    topics = random.sample(field_data["topics"], min(2, len(field_data["topics"])))
    dts    = random.sample(dates, 3)
    tms    = random.sample(times, 3)

    return (text
        .replace("{role}",         random.choice(field_data["roles"]))
        .replace("{tech2}",        techs[1])
        .replace("{tech}",         techs[0])
        .replace("{topic2}",       topics[1])
        .replace("{topic}",        topics[0])
        .replace("{format}",       random.choice(field_data["formats"]))
        .replace("{company}",      company)
        .replace("{seniority}",    random.choice(seniority_phrases))
        .replace("{sender_title}", random.choice(sender_titles))
        .replace("{interviewer}",  random.choice(interviewer_refs))
        .replace("{process}",      random.choice(process_terms))
        .replace("{platform}",     random.choice(platforms))
        .replace("{duration}",     random.choice(durations))
        .replace("{location}",     random.choice(locations))
        .replace("{deadline}",     random.choice(near_deadlines))
        .replace("{date3}",        dts[2]).replace("{date2}", dts[1]).replace("{date}", dts[0])
        .replace("{time3}",        tms[2]).replace("{time2}", tms[1]).replace("{time}", tms[0])
    )


def build_email(template, field_data, urgency):
    """
    Assemble a full email: subject, greeting, body, optional logistics, sign-off.

    Target median 75-90 words with a subject line. The previous dataset had a
    median of 34 words and NO subject lines at all, while real pasted email runs
    80+ with subject, signature and a logistics paragraph — a shape difference
    that on its own put a third of every real email outside the vocabulary.
    """
    company   = random.choice(companies)
    recruiter = random.choice(recruiter_names)
    name      = random.choice(first_names)

    parts = []
    parts.append("Subject: " + random.choice(template["subjects"]))
    parts.append("")
    parts.append(f"{random.choice(greetings)} {name},")
    parts.append("")
    parts.append(template["body"])

    if random.random() < 0.55:
        parts.append("")
        parts.append(random.choice(tech_context_lines))

    if template["logistics"] and random.random() < 0.90:
        parts.append("")
        parts.append(random.choice(template["logistics"]))

    # ALWAYS present: this sentence IS the urgency label's evidence. A row without
    # one would be an unlabelled example — a reader could not recover the answer
    # from the text, so neither could the model.
    parts.append("")
    parts.append(random.choice(urgency_markers[urgency]))

    # Dropped from 0.80 to compensate for the marker's length.
    if random.random() < 0.55:
        parts.append("")
        parts.append(random.choice(courtesy_lines))

    parts.append("")
    parts.append(random.choice(closing_lines))

    parts.append("")
    parts.append(f"{random.choice(signoffs)},")
    parts.append(recruiter)
    if random.random() < 0.60:
        parts.append(f"{random.choice(sender_titles)}, {company}")

    return fill_slots("\n".join(parts), field_data, company)


---
# NOISE
# Real pasted email is messier than a template
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# NOISE
# ══════════════════════════════════════════════════════════════════════════════
def maybe_add_noise(text):
    """
    Real pasted email is messier than a template. People drop the subject when
    copying the body, forward threads, strip signatures, and write in lower case
    on their phone.
    """
    lines = text.split("\n")

    # paste without the subject line
    if random.random() < 0.20 and lines and lines[0].startswith("Subject:"):
        lines = lines[2:] if len(lines) > 2 else lines
        text = "\n".join(lines)

    # drop the greeting line
    if random.random() < 0.15:
        lines = text.split("\n")
        for i, l in enumerate(lines):
            if l.endswith(",") and len(l.split()) <= 4:
                del lines[i]
                if i < len(lines) and lines[i].strip() == "":
                    del lines[i]
                break
        text = "\n".join(lines)

    # strip the signature block
    if random.random() < 0.20:
        for marker in ("\nBest,", "\nBest regards,", "\nKind regards,", "\nRegards,",
                       "\nMany thanks,", "\nThanks,", "\nAll the best,",
                       "\nWarm regards,", "\nSincerely,", "\nCheers,"):
            if marker in text:
                text = text.split(marker)[0]
                break

    # forwarded-thread header
    if random.random() < 0.08:
        text = "---------- Forwarded message ----------\n" + text

    # typed on a phone
    if random.random() < 0.07:
        text = text.lower()

    # shorthand substitutions
    if random.random() < 0.20:
        text = text.replace("Please let me know", "Let me know", 1)
    if random.random() < 0.15:
        text = text.replace("Thank you for", "Thanks for", 1)
    if random.random() < 0.15:
        text = text.replace("I would", "I'd", 1)

    return text


---
# GENERATE DATASET
---


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GENERATE
# ══════════════════════════════════════════════════════════════════════════════
random.seed(42)

templates = {**templates_a, **templates_b}

EXAMPLES_PER_COMBO = 24   # 7 categories x 11 fields x 24 = 1,848 rows

rows = []
for category, tmpl_list in templates.items():
    for job_field, field_data in job_fields.items():
        for _ in range(EXAMPLES_PER_COMBO):
            idx      = random.randrange(len(tmpl_list))
            template = tmpl_list[idx]

            # Urgency is drawn BEFORE the email is built so the marker sentence can
            # be chosen to match it. The label and its evidence are then the same
            # fact rather than two independent draws.
            urgency = choose_weighted(urgency_weights[category])

            text = build_email(template, field_data, urgency)
            text = maybe_add_noise(text)

            rows.append({
                "text":        text,
                "category":    category,
                "urgency":     urgency,
                "job_field":   job_field,
                # Grouping unit for train.py. Rows sharing this are fills of one
                # template and must never straddle the train/test split.
                "template_id": f"{category}#{idx}",
            })

random.shuffle(rows)

df = pd.DataFrame(rows)
df.to_csv(file_path, index=False)

print(f"\nDataset created: {df.shape} rows/columns")
print("─" * 45)
print(f"\n→ Templates: {df['template_id'].nunique()}")
print(f"→ Rows per template: median {int(df['template_id'].value_counts().median())}")
print("\n→ Category distribution:")
print(df["category"].value_counts())
print("\n→ Urgency distribution:")
print(df["urgency"].value_counts())
print("\n→ Job field distribution:")
print(df["job_field"].value_counts())
wc = df["text"].str.split().str.len()
print(f"\n→ Word count: min {wc.min()} | median {int(wc.median())} | max {wc.max()}")
print(f"→ Rows with a subject line: {df['text'].str.contains('Subject:').mean() * 100:.0f}%")
print("\n─── Preview ───")
print(df["text"].iloc[0])
